# NeoTwin: 02 — 3DGS Training (Mip-Splatting)

**Runtime:** Google Colab T4 GPU  
**Time:** ~12–18 min  
**Input:** `sparse_output.zip` from Notebook 01  
**Output:** `point_cloud.ply` + per-iteration PSNR/SSIM metrics

### Upgrades over baseline
| Setting | Baseline 3DGS | This notebook |
|---|---|---|
| Renderer | Standard splat | **Mip-Splatting** (anti-aliasing) |
| Iterations | 30 000 flat | **15 000 + eval every 3 k** (stops if converged) |
| Camera model | SIMPLE_PINHOLE | **OPENCV** (from NB01) |
| Quality check | None | PSNR ≥ 28 dB gate before download |
| Densification | Default | Tuned `--densify_grad_threshold 0.0002` |

In [ ]:
# ─── 0. CONFIG ───────────────────────────────────────────────────────────────
MAX_ITERATIONS      = 15000   # enough for convergence on most scenes
EVAL_EVERY          = 3000    # evaluate PSNR/SSIM at these checkpoints
EARLY_STOP_PSNR     = 31.0    # stop early if PSNR already exceeds this
MIN_ACCEPTABLE_PSNR = 27.0    # gate: refuse to download below this
DENSIFY_GRAD_THRESH = 0.0002  # tighter than default 0.0002 → more detail
USE_MIP_SPLATTING   = True    # always True for final builds
SCENE_NAME          = 'scene' # name used for output folder
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── 1. INSTALL ──────────────────────────────────────────────────────────────
import subprocess, sys
def run(cmd, **kw): return subprocess.run(cmd, shell=True, check=True, **kw)

run('apt-get update -qq && apt-get install -y -qq libglm-dev')
run('pip install -q plyfile torch torchvision tqdm pandas matplotlib')

if USE_MIP_SPLATTING:
    print('Cloning Mip-Splatting...')
    run('git clone -q https://github.com/autonomousvision/mip-splatting --recursive')
    run('pip install -q -e mip-splatting')
    REPO = 'mip-splatting'
else:
    print('Cloning standard 3DGS...')
    run('git clone -q https://github.com/graphdeco-inria/gaussian-splatting --recursive')
    run('pip install -q -e gaussian-splatting')
    REPO = 'gaussian-splatting'

print(f'✅ {REPO} ready')

In [ ]:
# ─── 2. UNPACK INPUT FROM NOTEBOOK 01 ────────────────────────────────────────
import shutil, os, json
from pathlib import Path
from google.colab import files

print('Upload sparse_output.zip from Notebook 01:')
uploaded = files.upload()
zip_name = next(iter(uploaded))

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
shutil.unpack_archive(zip_name, DATA_DIR)

# Verify structure
sparse_dirs = list((DATA_DIR / 'sparse').iterdir())
assert sparse_dirs, '❌ No sparse reconstruction found in zip'

# Read quality report if present
qr_path = DATA_DIR / 'quality_report.json'
if qr_path.exists():
    with open(qr_path) as f:
        qr = json.load(f)
    reg_rate = float(qr.get('registration_rate', '0%').replace('%', ''))
    assert reg_rate > 60, f'❌ Registration rate {reg_rate}% too low — re-run NB01'
    print(f'✅ NB01 quality report: {qr["images_registered"]} images registered ({reg_rate:.0f}%)')
else:
    print('⚠️  No quality_report.json found — proceeding without NB01 gate')

print(f'✅ Data ready at {DATA_DIR}')

In [ ]:
# ─── 3. TRAIN — with per-checkpoint evaluation ───────────────────────────────
import subprocess, re, json, time
from pathlib import Path

OUTPUT_DIR   = Path('output') / SCENE_NAME
METRICS_FILE = OUTPUT_DIR / 'training_metrics.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_log = []
best_psnr   = 0.0
converged   = False
checkpoints = list(range(EVAL_EVERY, MAX_ITERATIONS + 1, EVAL_EVERY))

train_script = f'{REPO}/train.py'

for checkpoint in checkpoints:
    if converged:
        break

    print(f'\n🚀 Training to iteration {checkpoint}/{MAX_ITERATIONS}...')
    t0 = time.time()

    cmd = [
        'python', train_script,
        '-s', str(DATA_DIR),
        '-m', str(OUTPUT_DIR),
        '--iterations', str(checkpoint),
        '--eval',
        '--densify_grad_threshold', str(DENSIFY_GRAD_THRESH),
        '--checkpoint_iterations', str(checkpoint),
        '--quiet'
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    # Parse PSNR from stdout
    psnr_match = re.search(r'PSNR[\s:=]+([\d.]+)', result.stdout)
    ssim_match = re.search(r'SSIM[\s:=]+([\d.]+)', result.stdout)
    psnr = float(psnr_match.group(1)) if psnr_match else None
    ssim = float(ssim_match.group(1)) if ssim_match else None

    entry = {
        'iteration': checkpoint,
        'psnr_db':   round(psnr, 2) if psnr else 'N/A',
        'ssim':      round(ssim, 4) if ssim else 'N/A',
        'time_sec':  round(elapsed, 1)
    }
    metrics_log.append(entry)

    print(f'   Iter {checkpoint:>6d} | PSNR: {entry["psnr_db"]} dB | SSIM: {entry["ssim"]} | {elapsed:.0f}s')

    if psnr and psnr > best_psnr:
        best_psnr = psnr
    if psnr and psnr >= EARLY_STOP_PSNR:
        print(f'   ⚡ Early stop: PSNR {psnr:.2f} dB ≥ {EARLY_STOP_PSNR} dB threshold')
        converged = True

# Save metrics
with open(METRICS_FILE, 'w') as f:
    json.dump(metrics_log, f, indent=2)

print(f'\n📊 Training complete. Best PSNR: {best_psnr:.2f} dB')

In [ ]:
# ─── 4. QUALITY GATE & PLOT ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import json
from pathlib import Path

with open(METRICS_FILE) as f:
    metrics_log = json.load(f)

iters  = [m['iteration'] for m in metrics_log]
psnrs  = [m['psnr_db'] if m['psnr_db'] != 'N/A' else None for m in metrics_log]
ssims  = [m['ssim']    if m['ssim']    != 'N/A' else None for m in metrics_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'3DGS Training — {SCENE_NAME}', fontsize=14)

if any(psnrs):
    ax1.plot(iters, psnrs, 'o-', color='#F7C131', linewidth=2)
    ax1.axhline(MIN_ACCEPTABLE_PSNR, color='red', linestyle='--', label=f'Min gate ({MIN_ACCEPTABLE_PSNR} dB)')
    ax1.axhline(EARLY_STOP_PSNR,     color='green', linestyle='--', label=f'Early stop ({EARLY_STOP_PSNR} dB)')
    ax1.set(xlabel='Iteration', ylabel='PSNR (dB)', title='PSNR over Training')
    ax1.legend(); ax1.grid(True, alpha=0.3)

if any(ssims):
    ax2.plot(iters, ssims, 'o-', color='#00CFFF', linewidth=2)
    ax2.set(xlabel='Iteration', ylabel='SSIM', title='SSIM over Training', ylim=(0, 1))
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# Quality gate
valid_psnrs = [p for p in psnrs if p is not None]
if valid_psnrs and max(valid_psnrs) < MIN_ACCEPTABLE_PSNR:
    raise AssertionError(
        f'❌ Best PSNR {max(valid_psnrs):.2f} dB < {MIN_ACCEPTABLE_PSNR} dB minimum.\n'
        '   Try: more images, better overlap, or longer training (raise MAX_ITERATIONS)'
    )
print('✅ Quality gate passed')

In [ ]:
# ─── 5. LOCATE & DOWNLOAD PLY ────────────────────────────────────────────────
import glob, shutil
from google.colab import files
from pathlib import Path

# Find the highest-iteration checkpoint .ply
ply_candidates = sorted(
    glob.glob(str(OUTPUT_DIR / 'point_cloud' / 'iteration_*' / 'point_cloud.ply')),
    key=lambda p: int(Path(p).parent.name.split('_')[1])
)
assert ply_candidates, '❌ No point_cloud.ply found — check training output'
best_ply = ply_candidates[-1]

# Bundle ply + metrics + training curve for NB03/NB04
bundle_dir = Path('neotwin_3dgs_output')
bundle_dir.mkdir(exist_ok=True)
shutil.copy(best_ply, bundle_dir / 'point_cloud.ply')
shutil.copy(METRICS_FILE, bundle_dir / 'training_metrics.json')
shutil.copy(OUTPUT_DIR / 'training_curve.png', bundle_dir / 'training_curve.png')

shutil.make_archive('neotwin_3dgs_output', 'zip', bundle_dir)

ply_size_mb = Path(best_ply).stat().st_size / 1e6
print(f'📦 Packaging: {best_ply}')
print(f'   PLY size: {ply_size_mb:.1f} MB')
print(f'   Best PSNR: {max(valid_psnrs):.2f} dB' if valid_psnrs else '')

files.download('neotwin_3dgs_output.zip')
print('✅ Download started → use this zip as input to Notebooks 03 and 04')